<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>Cómo funcionan las redes neuronales</b></font></h1>
<h2><b>Notebook 1: </b>Primera neurona</h2>
<br>
En este notebook entrenamos nuestra primera neurona para que haga una labor de predicción bastante particular...

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import utils

# Fijamos una seed para que sea reproducible los resultados, usando la respuesta de la vida (revisar Hitchhiker's Guide to the Galaxy)
torch.manual_seed(42)

----

<br>

## **Parte 0:** Nuestro objetivo

<div style="float: right; width: 36%; min-width: 170px; max-width: 300px; margin: 4px 0px 12px 30px;">
  <img src="assets/mensajes.png" width="100%" alt="Celular con mensajes sin responder"
       style="display: block; transform: rotate(-1.5deg);
              filter: drop-shadow(0px 12px 20px rgba(41, 196, 217, 0.35));"/>
</div>

Julián y Manuela llevan una relación feliz. Sin embargo, Manuela tiende a ser una persona muy atenta y demandante en la comunicación (intensa), por lo que a Julián le interesa anticipar cuántos mensajes de WhatsApp se le van a acumular mientras no contesta, con el objetivo de responder antes de que se arme un problema (tóxica).

El objetivo de este notebook es construir un modelo de redes neuronales capaz de predecir **cuántos mensajes habrá mandado Manuela** a partir de **los minutos que Julián lleva sin responder**, usando el historial de la conversación.

<div style="clear: both;"></div>

----

<br>

## **Parte 1:** Los datos

Empezaremos subiendo los datos sacados del historial de chat donde:

- **Entrada** ($x$): minutos que Julián lleva callado.
- **Salida** ($y$): mensajes que Manuela ha enviado hasta ese momento.

Queremos un modelo de la forma:

$$
F:x \rightarrow y
$$

que prediga cuántos mensajes enviará Manuela según el tiempo que Julián lleve sin responder.

In [ ]:
# El chat está en un csv con una columna de minutos y otra de mensajes
chat = pd.read_csv("data/mensajes.csv")

# Entrada (x): minutos que Julián lleva sin responder
minutos = torch.tensor(chat["minutos"].to_numpy(dtype="float32")).reshape(-1, 1)

# Salida esperada (y): mensajes que Manuela ya mandó
mensajes = torch.tensor(chat["mensajes"].to_numpy(dtype="float32")).reshape(-1, 1)

utils.plot_data(minutos, mensajes, title="Historial cantidad de mensajes")

Podemos notar un crecimiento preocupante de los mensajes a medida de que va pasando el tiempo.

----

<br>

## **Parte 2:** La neurona

Iniciaremos con una neurona simple, cuya ecuación tendrá la forma:

$$
y = \text{ReLU}(w \cdot x + b)
$$

Inicialmente, realizaremos la neurona a mano, y luego de dos formas usando Pytorch, una sencilla y otra usando clases personalizadas.


Inicialmente vamos a definir la función de activación RELU
$$
\text{ReLU}(z) = \max(0, z)
$$

In [ ]:
def relu(z):
    ### Inicio del código
    


    ### fin del código
    return output

Ahora vamos a definir el foward de una neurona
$$
\hat{y} = \text{ReLU}(w \cdot x + b)
$$

In [ ]:
def neurona(x, w, b):
    ### Inicio del código


    
    ### fin del código
    return ouput

Ahora podemos inicializar los pesos, lo haremos de forma aleatoria

In [ ]:
w = torch.randn(1)
b = torch.randn(1)

def predecir_mensajes(minutos_sin_responder):
    return neurona(minutos_sin_responder, w, b)

print(f"Parametros:\n peso w: {w} \n sesgo b: {b}")

utils.plot_fit(minutos, mensajes, predecir_mensajes, label="Neurona sin entrenar",title="La neurona antes de entrenar")

Ahora veamos el error usando mínimo cuadrados
$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2
$$

In [ ]:
def perdida_mse(y, y_pred):
    ### Inicio del código
    


    ### fin del código
    return output

perdida = perdida_mse(mensajes, neurona(minutos, w, b))

print(f"Pérdida (MSE):\n {perdida}")

Vemos que el error está bastante alto, por lo tanto ya lo que toca es optimizar, esto será usando el gradiente de *w* y el de *b*, definidos por:

**Gradiente de w**
$$
\frac{\partial \, \text{MSE}}{\partial \mathbf{w}} = \frac{d\,\text{MSE}()}{d\,\hat{y}} \cdot \frac{d\,\text{ReLU}()}{d\,z} \cdot \frac{\partial z}{\partial \mathbf{w}}
$$

**Gradiente de b**
$$
\frac{\partial \, \text{MSE}}{\partial b} = \frac{d\,\text{MSE}()}{d\,\hat{y}} \cdot \frac{d\,\text{ReLU}()}{d\,z} \cdot \frac{\partial z}{\partial b}
$$

Y acá todas las derivadas necesarias para realizar el cálculo:

| | |
|---|---|
| $$\dfrac{d\,\text{MSE}()}{d\,\hat{y}} = \dfrac{2}{n}(\hat{y} - y)$$ | $$\dfrac{d\,\text{ReLU}()}{d\,z} = \begin{cases} 1 & \text{si } z \ge 0 \\ 0 & \text{si } z < 0 \end{cases}$$ |
| $$\dfrac{\partial z}{\partial \mathbf{w}} = \mathbf{x}$$ | $$\dfrac{\partial z}{\partial b} = 1$$ |



In [ ]:
def dMSE(y_hat, y):
    ### Inicio del código




    ### fin del código
    return output

def dReLU(z):
    ### Inicio del código


        
    ### fin del código
    return output

def gradiente(x, y, w, b):
    ### Inicio del código
             

    
    ### fin del código
    return dw, db

In [ ]:
tasa = 0.01

historial_perdida = []

for epoca in range(2000):
    dw, db = gradiente(minutos, mensajes, w, b) 
    w = w - tasa * dw                           
    b = b - tasa * db                            

    historial_perdida.append(perdida_mse(mensajes, neurona(minutos, w, b)))
    if (epoca + 1) % 200 == 0:
        print(f"Época {epoca+1} — pérdida: {historial_perdida[-1]}")

utils.plot_loss(historial_perdida)

print(f"Parametros:\n peso w: {w} \n sesgo b: {b}")

utils.plot_fit(minutos, mensajes, lambda x: neurona(x, w, b), label="Neurona a mano", title="La neurona después de entrenar")

----

<br>

## **Parte 3:** Usando PyTorch

Vamos a rehacer la misma neurona de las dos formas que mencionamos.

La primera forma es bastante directa, es colocar de forma secuencial los cálculos que se harán en la neurona

In [ ]:
neurona_sequential = nn.Sequential(
    nn.Linear(1, 1), # w * x + b
    nn.ReLU()
)

Por otro lado usando herencia se puede realizar la misma red neuronal pero permitiendo en un futuro mucho más personalización

In [ ]:
class Neurona(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(1, 1) # w * x + b
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        return x

A partir de ahora usaremos esta segunda forma en todo el notebook

Ahora vamos a inicializar el modelo, definir su función de pérdida y su optimizador.

In [ ]:
modelo = Neurona()
funcion_perdida = nn.MSELoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.05)

Ahora sí, comenzaremos con el proceso de entrenamiento, el cual sigue el siguiente flujo:

**Borrar gradientes → Predecir → Medir error → Calcular gradientes → Actualizar parámetros → Repetir**

Este ciclo se repite durante varias épocas (*epochs*) que fueron definidas.

In [ ]:
historial_neurona = []

for epoca in range(2000):
    optimizador.zero_grad()                            # 1. borrar los gradientes viejos
    predicciones = modelo(minutos)                     # 2. predecir
    perdida = funcion_perdida(predicciones, mensajes)  # 3. medir el error
    perdida.backward()                                 # 4. calcular gradientes
    optimizador.step()                                 # 5. actualizar parametros

    historial_neurona.append(perdida.item())
    if (epoca + 1) % 200 == 0:
        print(f"Época {epoca + 1:4d} — pérdida: {perdida.item():.4f}")

# Graficando la perdida
utils.plot_loss(historial_neurona)

Ya tenemos el modelo entrenado, ahora veamos cómo quedó la función

In [ ]:
utils.plot_fit(minutos, mensajes, modelo, label="Neurona",title="Predicción de una sola neurona")

Julián al ver este modelo nos miró raro, por lo tanto tocará hacerle unos ajustes (poner más neuronas).

Representación de la mirada de Julián

![julian](assets/julian.png)

----

<br>

## **Parte 4:** La red neuronal

Ya tuvimos un primer intento de modelo usando una neuronal, que dió como resultado una recta con un doblez, pero eso no es suficiente.

Ahora usamos varias neuronas en una capa oculta, más una capa que combina sus salidas. Cada neurona aporta un doblez y juntas lograrán aproximar una mejor curva.

In [ ]:
class RedNeuronal(nn.Module):

    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(1, 5)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(5, 1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        return x

In [ ]:

modelo = RedNeuronal()
funcion_perdida = nn.MSELoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.05)

historial_red = []

for epoca in range(2000):
    optimizador.zero_grad()                            # 1. borrar los gradientes viejos
    predicciones = modelo(minutos)                     # 2. predecir
    perdida = funcion_perdida(predicciones, mensajes)  # 3. medir el error
    perdida.backward()                                 # 4. calcular gradientes
    optimizador.step()                                 # 5. actualizar parametros

    historial_red.append(perdida.item())

    if (epoca + 1) % 200 == 0:
        print(f"Época {epoca + 1} — pérdida: {perdida}")

# Graficando la perdida
utils.plot_loss(historial_red)

El modelo se ve con una buena curva de aprendizaje, ahora veamos que tal aproxima los datos

In [ ]:
utils.plot_fit(minutos, mensajes, modelo, label="Red neuronal",title="Predicción de la red neuronal")

Con este modelo, Julián ya puede aproximar cuántos mensajes le mandará Manuela según el tiempo que lleve sin responder. Así podrá anticipar el momento crítico, justo antes del desastre, y responder a tiempo para evitar una pelea.

-----

<br>

### **Siguiente notebook:**

[`02_clasificador_titanic.ipynb`](02_clasificador_titanic.ipynb)

### <font color="#29c4d9">**Notebook 1 listo.**</font>

<br>

---
<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>